# NATURAL PROCESSING LANGUAGE

**NAMA:** Lusi Fitriani

**NIM:** 25146065

# 1. Instalasi dan Import Pustaka

Berikut adalah cara untuk mengimport library yang diperlukan untuk menampilkan dataset dari Hugging Face serta melakukan case folding, cleaning, tokenisasi, stopword removal, dan stemming bahasa Indonesia. 



In [15]:
!pip install -q datasets pandas nltk PySastrawi

import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from datasets import load_dataset
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


True

# 2. Menampilkan Informasi Dataset

Berikut cara menampilkan jumlah data, nama kolom, tipe data, serta distribusi label sentimen. 

In [16]:
dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment")
df = dataset["train"].to_pandas()

print("=== INFORMASI DATASET ===")
print(f"Jumlah baris data : {len(df)}")
print(f"Jumlah Kolom      : {(df.shape)[1]}")
print(f"Daftar Kolom      : {list(df.columns)}")
print(f"\nTipe data setiap kolom :")
print(df.dtypes)

print("\n=== DISTRIBUSI LABEL SENTIMEN ===")
distribusi = df["sentimen"].value_counts().reset_index()
distribusi.columns = ["Label Sentimen", "Jumlah Data"]
distribusi["Persentase (%)"] = (distribusi["Jumlah Data"] / len(df) * 100).round(2)
print(distribusi.to_string(index=False))


=== INFORMASI DATASET ===
Jumlah baris data : 11606
Jumlah Kolom      : 3
Daftar Kolom      : ['review', 'translate', 'sentimen']

Tipe data setiap kolom :
review         str
translate      str
sentimen     int64
dtype: object

=== DISTRIBUSI LABEL SENTIMEN ===
 Label Sentimen  Jumlah Data  Persentase (%)
              0         5803            50.0
              1         5803            50.0


# 3. Implementasi Tahapan Preprocessing Teks

Tahapan preprocessing meliputi:

1. **Case Folding**     : mengubah teks menjadi huruf kecil.
2. **Cleaning**         : menghapus URL, angka, tanda baca, emoji, dan karakter khusus.
3. **Tokenisasi**       : memisahkan  teks menjadi daftar token kata.
4. **Stopword Removal** : menghapus kata tidak penting bahasa Indonesia.
5. **Stemming**         : mengubah kata berimbuhan menjadi bentuk kata dasar.



In [22]:
stopword_factory = StopWordRemoverFactory()
stopwords_id = set(stopword_factory.get_stop_words())

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def case_folding(text):
    return str(text).lower()

def cleaning(text):
    text = re.sub(r"http\S|www\.\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenization(text):
    return word_tokenize(text)

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stopwords_id]

def stemming(tokens):
    return [stemmer.stem(token) for token in tokens]

def preprocess_text(text):
    folded = case_folding(text)
    cleaned = cleaning(folded)
    tokens = tokenization(cleaned)
    no_stop = remove_stopwords(tokens)
    stemmed = stemming(no_stop)
    return stemmed


# 4. Hasil dari 10 Contoh Teks Sebelum dan Sesudah Preprocessing

Saya mengambil 10 sampel ulasan dari dataset dan memunculkan perbandingan antara teks asli sebelum preprocessing dengan hasil akhir setelah melewati seluruh tahapan .

In [23]:
sample_indices = [0, 1, 2, 3, 5, 11, 14, 15, 42, 50]
sample_df = df.iloc[sample_indices][["review", "sentimen"]].copy()
sample_df["case_folding"] = sample_df["review"].apply(case_folding)
sample_df["cleaning"] = sample_df["case_folding"].apply(cleaning)
sample_df["tokenisasi"] = sample_df["cleaning"].apply(tokenization)
sample_df["stopword_removal"] = sample_df["tokenisasi"].apply(remove_stopwords)
sample_df["stemming"] = sample_df["stopword_removal"].apply(stemming)
sample_df["hasil_akhir"] = sample_df["stemming"].apply(lambda tokens: " ".join(tokens))

for i, (_, row) in enumerate(sample_df.iterrows(), start=1):
    print(f"--- Contoh {i} (Label Sentimen: {row['sentimen']}) ---")
    print(f"Sebelum : {row['review']}")
    print(f"Sesudah : {row['hasil_akhir']}\n")

--- Contoh 1 (Label Sentimen: 0) ---
Sebelum : Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala
Sesudah : suara mic puas pesan yg nyala sy pasti lg klik yg nyala tp yg ga nyala

--- Contoh 2 (Label Sentimen: 1) ---
Sebelum : Good. Mudah-mudahan awet. Makasih 
Sesudah : good mudahmudahan awet makasih

--- Contoh 3 (Label Sentimen: 0) ---
Sebelum : kondisi dus robek2, tolong di perhatikan lagi
Sesudah : kondisi dus robek tolong perhati

--- Contoh 4 (Label Sentimen: 0) ---
Sebelum : Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(
Sesudah : kualitas produk harga jangkau kirim cepet banget udh order oke tp sayang klo kado gabisa nitip pake kartu ucap bungkus pake kertas kado

--- Contoh 5 (Label Sentimen: 1) ---
Sebelum : Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman

# 5. Pengaruh Preprocessing Terhadap Kualitas Data Teks:

1. **Case Folding** membuat data konsisten, kata Bagus dan bagus tidak lagi dianggap kata berbeda, sehingga jumlah kosa kata berkurang.

2. **Cleaning** menghilangkan noise seperti URL, angka, emoji, dan tanda baca yang tidak berpengaruh pada sentimen, sehingga model fokus pada kata yang bermakna.

3. **Tokenisasi** memecah kalimat menjadi kata-kata satuan agar mudah dihitung dan diolah menjadi fitur (misal TF-IDF).

4. **Stopword Removal** menghapus kata umum yang, dan, di, ini yang sering muncul tapi tidak membawa sentimen, sehingga data lebih ringan dan akurat.

5. **Stemming** menyatukan kata berimbuhan seperti pengiriman, dikirim, mengirim menjadi satu kata dasar kirim, sehingga model tidak bingung dan data menjadi lebih padat.

Secara keseluruhan, preprocessing membuat data lebih bersih, seragam, dan kecil ukurannya, sehingga model sentimen dapat belajar lebih cepat dan hasilnya lebih akurat.